In [1]:
# --- 1. Instalación de librerías ---
#  - TensorFlow 2.17+ (compatible con Keras 3)
#  - Keras 3 (API independiente para la construccion de redes neuronales)
#  - HGQ2 (paquete 'hgq2' -> import 'hgq')
#  - hls4ml (>=1.2.0, ya con soporte para Keras v3 y HGQ2)
#  - kagglehub (descarga del dataset desde Kaggle)
#  - Utilidades varias (scikit-learn, h5py, pandas, matplotlib, tqdm)

!pip install -q --upgrade pip

# Versión "razonable" y estable de las librerías clave.
!pip install -q "tensorflow>=2.17" "keras>=3.5.0"
!pip install -q "hgq2>=0.1.2"
!pip install -q "git+https://github.com/fastmachinelearning/hls4ml.git@main"
!pip install -q kagglehub scikit-learn h5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# --- 2. Imports y configuración global de entorno ---

import os
import random
import re
from pathlib import Path
from multiprocessing import cpu_count, Pool

import numpy as np
import pandas as pd
import h5py as h5
from tqdm import tqdm
from matplotlib import pyplot as plt

# Reducir el ruido de logs de TensorFlow
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Muy importante para Keras 3: indicamos que el backend es TensorFlow
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras

# ============================
# HGQ2 y hls4ml
# ============================
from hgq.layers import QDense           # capas densas cuantizadas
from hgq.config import QuantizerConfigScope, LayerConfigScope
from hgq.utils import trace_minmax      # calibración para hls4ml
from hgq.utils.sugar import FreeEBOPs   # callback para monitorizar EBOPs durante el entrenamiento

import hls4ml

def set_seed(seed: int = 42):
    """Fija semillas para intentar reproducibilidad razonable."""
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    # Deshabilita TF32 en GPU para evitar diferencias numéricas grandes
    try:
        tf.config.experimental.enable_tensor_float_32_execution(False)
    except Exception as e:
        print("Aviso: no se pudo desactivar TF32:", e)

set_seed(42)

print("Backend Keras:", keras.backend.backend())
print("TensorFlow version:", tf.__version__)


Backend Keras: tensorflow
TensorFlow version: 2.19.0


In [3]:
# --- 3. Descarga del dataset de jets desde Kaggle ---
# Dataset: "hls4ml-lhc-jet-dataset-150-particles"
#   - Contiene jets con:
#       * 'jets'               -> características a nivel de jet (HLF + otras)
#       * 'jetFeatureNames'    -> nombres de las features
#       * 'jetConstituentList' -> lista de hasta 150 partículas por jet

import kagglehub

# Descarga (o reusa caché local si ya está descargado)
dataset_root = Path(
    kagglehub.dataset_download("calad0i/hls4ml-lhc-jet-dataset-150-particles")
)
print("Carpeta raíz del dataset descargado:", dataset_root)

# Rutas a train/val dentro del dataset
train_path_root = dataset_root / "hls4ml_LHCjet_150p_train" / "train"
test_path_root  = dataset_root / "hls4ml_LHCjet_150p_val" / "val"

print("\nArchivos encontrados (vista rápida):")
for p in sorted(dataset_root.rglob("*.h5"))[:10]:
    print("  ", p)


100%|██████████| 3.61G/3.61G [02:05<00:00, 30.8MB/s]

Extracting files...


Carpeta raíz del dataset descargado: /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-particles/versions/1

Archivos encontrados (vista rápida):
   /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-particles/versions/1/hls4ml_LHCjet_150p_train/train/jetImage_0_150p_0_10000.h5
   /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-particles/versions/1/hls4ml_LHCjet_150p_train/train/jetImage_0_150p_10000_20000.h5
   /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-particles/versions/1/hls4ml_LHCjet_150p_train/train/jetImage_0_150p_20000_30000.h5
   /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-particles/versions/1/hls4ml_LHCjet_150p_train/train/jetImage_0_150p_30000_40000.h5
   /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-particles/versions/1/hls4ml_LHCjet_150p_train/train/jetImage_0_150p_40000_50000.h5
   /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-partic

In [4]:
# --- 4. Exploración de un archivo HDF5 ---
# Tomamos el primer archivo de entrenamiento como ejemplo
train_paths = sorted(train_path_root.glob("**/*.h5"))
test_paths  = sorted(test_path_root.glob("**/*.h5"))

print(f"Nº de archivos de entrenamiento: {len(train_paths)}")
print(f"Nº de archivos de validación/test: {len(test_paths)}")

if len(train_paths) == 0:
    raise RuntimeError("No se encontraron archivos .h5 en train_path_root.")

example_file = train_paths[0]
print("\nEjemplo de archivo:", example_file)

with h5.File(example_file, "r") as f:
    print("\nContenido top-level del archivo:")
    for name, obj in f.items():
        try:
            print(f" - {name:25s} shape={obj.shape}")
        except Exception:
            print(f" - {name:25s} (sin atributo shape)")

    # Extraemos los nombres de características de jet
    raw_names = f["jetFeatureNames"]
    # Pueden venir como bytes -> los decodificamos a str
    jet_fea_names = []
    for entry in raw_names:
        if isinstance(entry, bytes):
            jet_fea_names.append(entry.decode("utf-8"))
        else:
            jet_fea_names.append(str(entry))

print("\nLista completa de nombres de características (jetFeatureNames):")
for i, name in enumerate(jet_fea_names):
    print(f"{i:2d} {name:25s}", end="")
    if (i + 1) % 4 == 0:
        print()
if len(jet_fea_names) % 4 != 0:
    print()  # salto de línea final


Nº de archivos de entrenamiento: 62
Nº de archivos de validación/test: 26

Ejemplo de archivo: /root/.cache/kagglehub/datasets/calad0i/hls4ml-lhc-jet-dataset-150-particles/versions/1/hls4ml_LHCjet_150p_train/train/jetImage_0_150p_0_10000.h5

Contenido top-level del archivo:
 - jetConstituentList        shape=(10000, 150, 16)
 - jetFeatureNames           shape=(59,)
 - jetImage                  shape=(10000, 100, 100)
 - jetImageECAL              shape=(10000, 100, 100)
 - jetImageHCAL              shape=(10000, 100, 100)
 - jets                      shape=(10000, 59)
 - particleFeatureNames      shape=(17,)

Lista completa de nombres de características (jetFeatureNames):
 0 j_ptfrac                  1 j_pt                      2 j_eta                     3 j_mass                   
 4 j_tau1_b1                 5 j_tau2_b1                 6 j_tau3_b1                 7 j_tau1_b2                
 8 j_tau2_b2                 9 j_tau3_b2                10 j_tau32_b1               11 j_tau32

In [5]:
# --- 5. Selección de las HLF que vamos a usar ---
# Nombres "base" de las HLF (sin prefijo j_)
# Referencia: dataset hls4ml_lhc_jets_hlf (16 features)
#   zlogz, c1_b0_mmdt, ..., mass_mmdt, multiplicity
hlf_base_names = [
    "zlogz",
    "c1_b0_mmdt", "c1_b1_mmdt", "c1_b2_mmdt",
    "c2_b1_mmdt", "c2_b2_mmdt",
    "d2_b1_mmdt", "d2_b2_mmdt",
    "d2_a1_b1_mmdt", "d2_a1_b2_mmdt",
    "m2_b1_mmdt", "m2_b2_mmdt",
    "n2_b1_mmdt", "n2_b2_mmdt",
    "mass_mmdt",
    "multiplicity",
]

# En este dataset, las features de jet vienen como 'j_<nombre>'
use_features = [f"j_{name}" for name in hlf_base_names]

# Comprobamos que existen en jet_fea_names
missing = [name for name in use_features if name not in jet_fea_names]
if missing:
    print("ERROR: Algunas HLF no se encuentran en jetFeatureNames:")
    for name in missing:
        print("  -", name)
    raise KeyError("Revisa que el dataset es el correcto.")

# Índices de las columnas correspondientes en el array 'jets'
use_idxs = np.array([jet_fea_names.index(name) for name in use_features], dtype=int)

print("\nHLF seleccionadas y sus índices en 'jets':")
for nm, idx in zip(use_features, use_idxs):
    print(f" - {nm:25s} -> columna {idx}")
print("\nTotal de HLF seleccionadas:", len(use_idxs))



HLF seleccionadas y sus índices en 'jets':
 - j_zlogz                   -> columna 12
 - j_c1_b0_mmdt              -> columna 34
 - j_c1_b1_mmdt              -> columna 35
 - j_c1_b2_mmdt              -> columna 36
 - j_c2_b1_mmdt              -> columna 37
 - j_c2_b2_mmdt              -> columna 38
 - j_d2_b1_mmdt              -> columna 39
 - j_d2_b2_mmdt              -> columna 40
 - j_d2_a1_b1_mmdt           -> columna 41
 - j_d2_a1_b2_mmdt           -> columna 42
 - j_m2_b1_mmdt              -> columna 43
 - j_m2_b2_mmdt              -> columna 44
 - j_n2_b1_mmdt              -> columna 45
 - j_n2_b2_mmdt              -> columna 46
 - j_mass_mmdt               -> columna 48
 - j_multiplicity            -> columna 52

Total de HLF seleccionadas: 16


In [6]:
# --- 6. Carga de HLF y etiquetas desde los archivos .h5 ---

def load_hlf_from_path(path: Path):
    """
    Carga las high level features (HLF) y las etiquetas de un archivo .h5.

    Parámetros
    ----------
    path : Path
        Ruta al archivo .h5 con grupos/datasets:
          - 'jets' (2D array)
          - posiblemente 'jetConstituentList', etc.

    Returns
    -------
    fea : np.ndarray
        Array de forma (n_jets, n_hlf_seleccionadas).
    label : np.ndarray
        Etiquetas enteras (0..4), de forma (n_jets,).
    """
    with h5.File(path, "r") as f:
        jets = np.asarray(f["jets"])  # (N, num_features_total)
        # Extraemos solo las columnas de HLF que queremos
        fea = jets[:, use_idxs]

        # La codificación de la etiqueta suele estar en las últimas 5 columnas
        # en one-hot: jets[:, -6:-1] con las 5 últimas válidas.
        lbl_oh = jets[:, -6:-1]

        # Comprobamos integridad one-hot: debe haber exactamente un '1' por fila
        if not np.all(lbl_oh.sum(axis=1) == 1):
            raise ValueError(f"One-hot inválido en el archivo: {path}")

        label = np.argmax(lbl_oh, axis=1).astype(np.int32)
    return fea.astype(np.float32), label


# Carga de TODOS los archivos de train
print("Cargando HLF de entrenamiento...")
with Pool(cpu_count()) as pool:
    iterator = pool.imap(load_hlf_from_path, train_paths)
    fea_list, label_list = zip(*tqdm(iterator, total=len(train_paths)))

train_fea = np.ascontiguousarray(np.concatenate(fea_list).astype(np.float32))
train_label = np.ascontiguousarray(np.concatenate(label_list).astype(np.int32))

print("Cargando HLF de test/validación...")
with Pool(cpu_count()) as pool:
    iterator = pool.imap(load_hlf_from_path, test_paths)
    fea_list, label_list = zip(*tqdm(iterator, total=len(test_paths)))

test_fea = np.ascontiguousarray(np.concatenate(fea_list).astype(np.float32))
test_label = np.ascontiguousarray(np.concatenate(label_list).astype(np.int32))

print("\nShapes finales (antes de split train/val):")
print("train_fea :", train_fea.shape)
print("train_label:", train_label.shape)
print("test_fea  :", test_fea.shape)
print("test_label:", test_label.shape)


Cargando HLF de entrenamiento...


100%|██████████| 62/62 [00:01<00:00, 37.84it/s]


Cargando HLF de test/validación...


100%|██████████| 26/26 [00:00<00:00, 36.46it/s]


Shapes finales (antes de split train/val):
train_fea : (620000, 16)
train_label: (620000,)
test_fea  : (260000, 16)
test_label: (260000,)


In [7]:
# --- 7. Split train/val y normalización de features ---

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split de validación (15% del train)
train_fea, val_fea, train_label, val_label = train_test_split(
    train_fea,
    train_label,
    test_size=0.15,
    shuffle=True,
    random_state=42,
    stratify=train_label,  # asegura distribución similar de clases
)

print("Shapes tras el split train/val:")
print("train_fea:", train_fea.shape)
print("val_fea  :", val_fea.shape)
print("test_fea :", test_fea.shape)

# Estandarización: media 0, varianza 1
scaler = StandardScaler()
train_fea = scaler.fit_transform(train_fea)
val_fea   = scaler.transform(val_fea)
test_fea  = scaler.transform(test_fea)

print("\nEjemplo de media/varianza de las primeras columnas (train):")
print("mean:", train_fea.mean(axis=0)[:5])
print("std :", train_fea.std(axis=0)[:5])


Shapes tras el split train/val:
train_fea: (527000, 16)
val_fea  : (93000, 16)
test_fea : (260000, 16)

Ejemplo de media/varianza de las primeras columnas (train):
mean: [ 9.0301606e-09  9.9674917e-09  1.6597971e-09 -1.2277511e-08
 -6.9171926e-09]
std : [0.9999103  0.99992245 0.9998891  0.99989474 0.9999079 ]


In [8]:
# --- 8. Conversión a tensores de TensorFlow ---

train_fea_tf   = tf.convert_to_tensor(train_fea, dtype=tf.float32)
val_fea_tf     = tf.convert_to_tensor(val_fea, dtype=tf.float32)
test_fea_tf    = tf.convert_to_tensor(test_fea, dtype=tf.float32)

train_label_tf = tf.convert_to_tensor(train_label, dtype=tf.int32)
val_label_tf   = tf.convert_to_tensor(val_label, dtype=tf.int32)
test_label_tf  = tf.convert_to_tensor(test_label, dtype=tf.int32)

print("Tensores TF:")
print("train_fea_tf:", train_fea_tf.shape)
print("val_fea_tf  :", val_fea_tf.shape)
print("test_fea_tf :", test_fea_tf.shape)


Tensores TF:
train_fea_tf: (527000, 16)
val_fea_tf  : (93000, 16)
test_fea_tf : (260000, 16)


In [9]:
# --- 9. Definición del modelo HGQ2 (QDense) ---

def build_hlf_quantized_mlp(n_hlf: int, n_classes: int = 5) -> keras.Model:
    """
    Construye un MLP cuantizado para clasificación de jets usando HGQ2.

    Arquitectura:
      Input (n_hlf) ->
      QDense(64, ReLU) ->
      QDense(32, ReLU) ->
      QDense(32, ReLU) ->
      QDense(n_classes, logits)
    """

    # Configuración de cuantización recomendada por HGQ2 para hardware/hls4ml:
    #  - Pesos (place='all', 'kbi', SAT_SYM)
    #  - Activaciones (place='datalane', 'kif', WRAP)
    #  - EBOPs activados para monitorizar coste de recursos
    with (
        QuantizerConfigScope(place="all",      default_q_type="kbi", overflow_mode="SAT_SYM"),
        QuantizerConfigScope(place="datalane", default_q_type="kif", overflow_mode="WRAP"),
        LayerConfigScope(enable_ebops=True, beta0=1e-5)
    ):
        inp = keras.Input(shape=(n_hlf,), name="hlf_input")

        x = QDense(64, activation="relu", name="dense1")(inp)
        x = QDense(32, activation="relu", name="dense2")(x)
        x = QDense(32, activation="relu", name="dense3")(x)

        # Sin softmax -> usaremos SparseCategoricalCrossentropy(from_logits=True)
        logits = QDense(n_classes, activation=None, name="output")(x)

        model = keras.Model(inputs=inp, outputs=logits, name="HLF_QDense_MLP")

    return model


# Creamos el modelo
n_hlf = train_fea.shape[1]  # debe ser 16
model = build_hlf_quantized_mlp(n_hlf=n_hlf, n_classes=5)

# Resumen de la arquitectura
model.summary()


Model: "HLF_QDense_MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hlf_input (InputLayer)          │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (QDense)                 │ (None, 64)             │         4,403 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (QDense)                 │ (None, 32)             │         8,515 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense3 (QDense)                 │ (None, 32)             │         4,323 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (QDense)                 │ (None, 5)              │           759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,000 (57.03 KB)

 Trainable params: 13,311 (52.00 KB)

 Non-trainable params: 4,689 (5.04 KB)

In [10]:
# --- 10. Compilación y entrenamiento ---

batch_size = 33200   # batch grande (suele ir bien para HGQ)
learning_rate = 5e-3
max_epochs = 200      # mayores epocas para comparacion

optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=["accuracy"],
)

# Callback para monitorizar EBOPs (coste de recursos estimado)
ebops_callback = FreeEBOPs()

history = model.fit(
    train_fea,            # se puede usar igualmente train_fea_tf
    train_label,
    validation_data=(val_fea, val_label),
    batch_size=batch_size,
    epochs=max_epochs,
    callbacks=[ebops_callback],
    verbose=1,
)

print("Entrenamiento completado.")


Epoch 1/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 17s 519ms/step - accuracy: 0.4975 - loss: 2.3050 - val_accuracy: 0.5827 - val_loss: 1.1124 - ebops: 102720.0000
Epoch 2/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 306ms/step - accuracy: 0.5918 - loss: 2.1000 - val_accuracy: 0.6193 - val_loss: 1.0055 - ebops: 102720.0000
Epoch 3/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 304ms/step - accuracy: 0.6561 - loss: 1.9947 - val_accuracy: 0.6721 - val_loss: 0.9550 - ebops: 102720.0000
Epoch 4/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 356ms/step - accuracy: 0.6865 - loss: 1.9403 - val_accuracy: 0.6981 - val_loss: 0.8765 - ebops: 102720.0000
Epoch 5/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 348ms/step - accuracy: 0.6991 - loss: 1.9006 - val_accuracy: 0.7072 - val_loss: 0.8472 - ebops: 102720.0000
Epoch 6/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 379ms/step - accuracy: 0.6885 - loss: 1.9135 - val_accuracy: 0.6552 - val_loss: 0.9513 - ebops: 102868.0000
Epoch 7/200
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 287ms/step - accuracy: 0.6834 - loss: 1.8353 - val_accura

KeyboardInterrupt: 

In [ ]:
# --- 11. Visualización de métricas de entrenamiento ---

history_dict = history.history

plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history_dict["loss"], label="train_loss")
plt.plot(history_dict["val_loss"], label="val_loss")
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.grid(True)
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(history_dict["accuracy"], label="train_acc")
plt.plot(history_dict["val_accuracy"], label="val_acc")
plt.xlabel("Época")
plt.ylabel("Exactitud")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# --- 12. Evaluación en test ---

test_loss, test_acc = model.evaluate(test_fea, test_label, verbose=0)
print(f"Pérdida en test:  {test_loss:.4f}")
print(f"Accuracy en test: {test_acc:.4f}")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

#Predicciones del modelo en el conjunto de test
X_test_for_model = test_fea  # tensor o array de forma (N_test, n_hlf)

# Para un modelo de dos entradas (celda 9),
# X_test_for_model = [test_fea, X_test_constituents]

# Ejecutar predicción
y_test_proba = model.predict(X_test_for_model, batch_size=4096)
# Clase predicha = índice de la probabilidad máxima
y_test_pred  = np.argmax(y_test_proba, axis=1)

# Convertir labels verdaderas a numpy por si siguen siendo tensores TF
try:
    y_test_true = test_label.numpy()
except AttributeError:
    y_test_true = np.asarray(test_label)

print("Shapes:")
print("  y_test_true:", y_test_true.shape)
print("  y_test_proba:", y_test_proba.shape)
print("  y_test_pred :", y_test_pred.shape)


In [ ]:
class_names = [
    "gluon (g)",        # 0
    "quark ligero (q)", # 1
    "W boson (w)",      # 2
    "Z boson (z)",      # 3
    "top quark (t)"     # 4
]

n_classes = len(class_names)
labels = list(range(n_classes))

cm = confusion_matrix(y_test_true, y_test_pred, labels=labels)

plt.figure(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(values_format="d", cmap="Blues", xticks_rotation=45)
plt.title("Matriz de confusión 5×5 (q/g/W/Z/t)")
plt.tight_layout()
plt.show()

print("\nReporte de clasificación por clase:")
print(classification_report(y_test_true, y_test_pred, target_names=class_names))


In [ ]:
def plot_binary_confusion(
    model,
    X_test_fea,
    y_test_true,
    class_a_idx,
    class_b_idx,
    class_names,
    X_test_constituents=None,
    title_prefix=""):
    """
    Construye una matriz de confusión 2×2 para discriminar entre dos tipos de jets.

    Parameters
    ----------
    model : tf.keras.Model
        Modelo entrenado (5 clases).
    X_test_fea : np.ndarray o tf.Tensor, shape (N, n_hlf)
        High-level features del conjunto de test.
    y_test_true : np.ndarray de enteros, shape (N,)
        Labels verdaderas (0..4).
    class_a_idx : int
        Índice entero de la clase A.
    class_b_idx : int
        Índice entero de la clase B.
    class_names : list[str]
        Lista con nombres legibles por índice.
    X_test_constituents : np.ndarray opcional
        Si tu modelo usa también constituyentes, pasa aquí el array (N, Npart, nfeats).
    title_prefix : str
        Texto para anteponer en el título de la figura.
    """

    # Seleccionar solo los jets que pertenecen a las dos clases de interés
    mask = np.isin(y_test_true, [class_a_idx, class_b_idx])
    y_pair_true = y_test_true[mask]

    if X_test_constituents is None:
        X_pair = X_test_fea[mask]
        model_input = X_pair
    else:
        X_pair_fea = X_test_fea[mask]
        X_pair_const = X_test_constituents[mask]
        model_input = [X_pair_fea, X_pair_const]

    # Predicciones de probabilidad (5 clases) y "colapso" a las dos clases de interés
    y_pair_proba = model.predict(model_input, batch_size=4096)
    # Quedarse solo con las columnas de las clases A y B
    pair_indices = np.array([class_a_idx, class_b_idx])
    proba_pair = y_pair_proba[:, pair_indices]

    # Predicción binaria: elegir la clase (A o B) con mayor probabilidad
    best_idx = np.argmax(proba_pair, axis=1)
    y_pair_pred = pair_indices[best_idx]

    # Construir matriz de confusión 2×2 en el orden [class_a, class_b]
    labels_pair = [class_a_idx, class_b_idx]
    cm_pair = confusion_matrix(y_pair_true, y_pair_pred, labels=labels_pair)

    display_labels = [class_names[i] for i in labels_pair]

    plt.figure(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_pair, display_labels=display_labels)
    disp.plot(values_format="d", cmap="Greens")
    plt.xticks(rotation=45)
    if title_prefix:
        plt.title(f"{title_prefix}: {display_labels[0]} vs {display_labels[1]}")
    else:
        plt.title(f"{display_labels[0]} vs {display_labels[1]}")
    plt.tight_layout()
    plt.show()

    print(f"\nReporte binario para {display_labels[0]} vs {display_labels[1]}:")
    print(classification_report(
        y_pair_true,
        y_pair_pred,
        target_names=display_labels
    ))


In [ ]:
X_test_constituents = None

# W vs quark (sensibilidad fuerte a masa + shape)
plot_binary_confusion(
    model=model,
    X_test_fea=test_fea.numpy() if hasattr(test_fea, "numpy") else test_fea,
    y_test_true=y_test_true,
    class_a_idx=2,   # W
    class_b_idx=1,   # quark
    class_names=class_names,
    X_test_constituents=X_test_constituents,
    title_prefix="W vs quark"
)

# W vs Z (diferencias muy sutiles, casi todo masa y detalles finos de shape)
plot_binary_confusion(
    model=model,
    X_test_fea=test_fea.numpy() if hasattr(test_fea, "numpy") else test_fea,
    y_test_true=y_test_true,
    class_a_idx=2,   # W
    class_b_idx=3,   # Z
    class_names=class_names,
    X_test_constituents=X_test_constituents,
    title_prefix="W vs Z"
)

# quark vs gluon (dominado por shape + multiplicidad)
plot_binary_confusion(
    model=model,
    X_test_fea=test_fea.numpy() if hasattr(test_fea, "numpy") else test_fea,
    y_test_true=y_test_true,
    class_a_idx=1,   # quark
    class_b_idx=0,   # gluon
    class_names=class_names,
    X_test_constituents=X_test_constituents,
    title_prefix="quark vs gluon"
)

# top vs gluon (análoga al apéndice A del paper 01)
plot_binary_confusion(
    model=model,
    X_test_fea=test_fea.numpy() if hasattr(test_fea, "numpy") else test_fea,
    y_test_true=y_test_true,
    class_a_idx=4,   # top
    class_b_idx=0,   # gluon
    class_names=class_names,
    X_test_constituents=X_test_constituents,
    title_prefix="top vs gluon"
)


In [ ]:
# --- 13. Calibración con trace_minmax ---

# Usamos una muestra de entrenamiento como "calibration set"
calib_samples = 50000
calib_data = train_fea[:calib_samples]

print(f"Calibrando trace_minmax con {calib_data.shape[0]} jets...")

trace_minmax(
    model,
    calib_data,
    verbose=True,
)

print("Calibración completada.")


In [ ]:
# 14. Conversión a hls4ml

from hls4ml.converters import convert_from_keras_model

# Directorio donde se generará el proyecto de hls4ml
output_dir = "hls4ml_prj_hgq2_jets"

hls_config = {
    # 'part': 'xcku115-flvb2104-2-i',      # ejemplo de FPGA
    # 'clock_period': 5,                   # en ns
    "backend": "Vitis",                    # Vitis o Vivado (HGQ2 soporta ambos)
    "io_type": "io_parallel",              # 'io_parallel' o 'io_stream'
}

# Nota importante:
#   - No se pasa (precision config),
#     HGQ2 ya transmite las precisiones necesarias para bit-exactitud.
hls_model = convert_from_keras_model(
    model,
    output_dir=output_dir,
    backend=hls_config["backend"],
    io_type=hls_config["io_type"],
)

print(f"Proyecto hls4ml creado en: {output_dir}")

# Compilamos el modelo HLS (traducción a C++ + build interno)
hls_model.compile()
print("Modelo hls4ml compilado.")


In [ ]:
# 15. Comparación Keras vs hls4ml (bit-exactitud)

# Tomamos algunos eventos de test
n_check = 1000
x_check = test_fea[:n_check]

keras_logits = model.predict(x_check, verbose=0)
hls_logits   = hls_model.predict(x_check)

# Comparamos clase predicha
keras_pred = np.argmax(keras_logits, axis=1)
hls_pred   = np.argmax(hls_logits, axis=1)

n_diff = np.count_nonzero(keras_pred != hls_pred)
print(f"Diferencias Keras vs hls4ml (argmax): {n_diff} / {n_check}")

# Diferencias absolutas en logits:
max_abs_diff = np.max(np.abs(keras_logits - hls_logits))
print(f"Máxima diferencia absoluta en logits: {max_abs_diff:.3e}")
